In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# Username and password are BOTH aacuser (per your note)
username = "aacuser"
password = "aacuser"

# Connect to database via CRUD Module
# NOTE: this assumes your AnimalShelter __init__ takes (username, password)
# If yours also requires host/port/db/collection, tell me and I’ll adjust.
db = AnimalShelter(username, password)

# Initial load (unfiltered)
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB ObjectId column to prevent Dash DataTable crash
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo (make sure this filename exists in your project folder)
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())

app.layout = html.Div([

    html.Center(html.B(html.H1("CS-340 Dashboard"))),

    # Logo (clickable) + Unique Identifier (your name)
    html.Center(
        html.Div([
            html.A(
                html.Img(
                    src="data:image/png;base64,{}".format(encoded_image.decode()),
                    style={"height": "150px"}
                ),
                href="https://www.snhu.edu",
                target="_blank"
            ),
            html.H4("Tom Pienkowski")
        ])
    ),

    html.Hr(),

    # Interactive filtering options
    html.Div([
        html.Label("Rescue Type Filter"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset", "value": "reset"},
            ],
            value="reset",
            inline=True
        ),
    ]),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "100px", "width": "150px", "maxWidth": "250px"},
    ),

    html.Br(),
    html.Hr(),

    # Chart + Map side-by-side
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", className="col s12 m6"),
            html.Div(id="map-id", className="col s12 m6"),
        ],
    ),
])


#############################################
# Interaction Between Components / Controller
#############################################

# Filter and update the data table from MongoDB
@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")


# Chart of your choice (Pie chart of breeds) driven by the table
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return [html.Div("No data to display.")]

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns:
        return [html.Div("Column 'breed' not found for chart.")]

    fig = px.pie(dff, names="breed", title="Breed Distribution (Filtered Results)")

    return [dcc.Graph(figure=fig)]


# Highlight selected columns in the table
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]

# Geo-location chart for the selected data entry
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available for map.")]

    dff = pd.DataFrame.from_dict(viewData)

    # Default row selection if none
    row = 0
    if index is not None and len(index) > 0:
        row = index[0]

    # Try to find lat/lon columns by common names
    lat_col = None
    lon_col = None

    for candidate in ["location_lat", "lat", "latitude"]:
        if candidate in dff.columns:
            lat_col = candidate
            break

    for candidate in ["location_long", "lon", "longitude", "long"]:
        if candidate in dff.columns:
            lon_col = candidate
            break

    if lat_col is None or lon_col is None:
        return [html.Div("Latitude/Longitude columns not found. Please check your dataset column names.")]

    # Tooltip and popup fields
    breed_val = dff.loc[row, "breed"] if "breed" in dff.columns else "Breed N/A"
    name_val = dff.loc[row, "name"] if "name" in dff.columns else "Name N/A"

    lat_val = dff.loc[row, lat_col]
    lon_val = dff.loc[row, lon_col]

    # Austin TX default center
    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat_val, lon_val],
                    children=[
                        dl.Tooltip(str(breed_val)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(name_val))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app (if port 8050 is busy, add: app.run_server(port=8051)
app.run_server()

Dash app running on https://oberonboxer-tunnelexplain-3000.codio.io/proxy/8050/
